# Rate Allocator Demo

Test the allocator with different scenarios and see how it distributes cash across tiered interest rates.

## Setup: Import and Load Data

In [ ]:
import json
from rate_allocator import allocate, Institution, Tier

# Load sample institutions from JSON
with open('data/sample_institutions.json') as f:
    data = json.load(f)

# Parse into Institution objects
institutions = []
for inst_data in data['institutions']:
    tiers = tuple(
        Tier(
            limit=float('inf') if t['limit'] == 'inf' else t['limit'],
            rate=t['rate']
        )
        for t in inst_data['tiers']
    )
    institutions.append(
        Institution(name=inst_data['name'], tiers=tiers)
    )

print(f"Loaded {len(institutions)} institutions")
for inst in institutions:
    print(f"  {inst.name}: {len(inst.tiers)} tiers")

## Scenario 1: Small Allocation (25,000 MXN)

This amount fits entirely in the first tier of Nu (highest rate).

In [ ]:
result = allocate(total=25_000, institutions=institutions)

print(f"\n=== Allocation for 25,000 MXN ===")
print(f"Effective rate: {result.effective_rate:.2%}")
print(f"Expected annual return: ${result.expected_return:,.2f}")
print(f"\nBy institution:")

for name, amounts in result.allocations.items():
    total_alloc = sum(amounts)
    if total_alloc > 0:
        print(f"  {name}: ${total_alloc:,.0f}")
        for i, amt in enumerate(amounts):
            if amt > 0:
                inst = next(i for i in institutions if i.name == name)
                rate = inst.tiers[i].rate
                print(f"    Tier {i+1}: ${amt:,.0f} @ {rate:.1%}")

## Scenario 2: Medium Allocation (100,000 MXN)

This tests how the allocator fills multiple institutions across their tier structures.

In [ ]:
result = allocate(total=100_000, institutions=institutions)

print(f"\n=== Allocation for 100,000 MXN ===")
print(f"Effective rate: {result.effective_rate:.2%}")
print(f"Expected annual return: ${result.expected_return:,.2f}")
print(f"\nBy institution:")

for name, amounts in result.allocations.items():
    total_alloc = sum(amounts)
    if total_alloc > 0:
        print(f"  {name}: ${total_alloc:,.0f}")
        for i, amt in enumerate(amounts):
            if amt > 0:
                inst = next(i for i in institutions if i.name == name)
                rate = inst.tiers[i].rate
                print(f"    Tier {i+1}: ${amt:,.0f} @ {rate:.1%}")

## Scenario 3: Large Allocation (500,000 MXN)

This tests how the allocator distributes across all available tiers.

In [ ]:
result = allocate(total=500_000, institutions=institutions)

print(f"\n=== Allocation for 500,000 MXN ===")
print(f"Effective rate: {result.effective_rate:.2%}")
print(f"Expected annual return: ${result.expected_return:,.2f}")
print(f"\nBy institution:")

for name, amounts in result.allocations.items():
    total_alloc = sum(amounts)
    if total_alloc > 0:
        print(f"  {name}: ${total_alloc:,.0f}")
        for i, amt in enumerate(amounts):
            if amt > 0:
                inst = next(i for i in institutions if i.name == name)
                rate = inst.tiers[i].rate
                print(f"    Tier {i+1}: ${amt:,.0f} @ {rate:.1%}")

## Scenario 4: Compare Different Amounts

See how the effective rate changes as the total amount increases.

In [ ]:
import pandas as pd

# Test different amounts
amounts = [10_000, 25_000, 50_000, 100_000, 250_000, 500_000, 1_000_000]
results = []

for amt in amounts:
    result = allocate(total=amt, institutions=institutions)
    results.append({
        'Total Amount': f'${amt:,}',
        'Effective Rate': f'{result.effective_rate:.2%}',
        'Expected Return': f'${result.expected_return:,.2f}'
    })

df = pd.DataFrame(results)
print("\n=== Effective Rate by Total Amount ===")
print(df.to_string(index=False))

## Scenario 5: Custom Institutions

Create your own institution structure and test the allocator.

In [ ]:
# Example: Create a custom institution
my_institutions = [
    Institution(
        name="My Bank A",
        tiers=(
            Tier(limit=50_000, rate=0.12),
            Tier(limit=200_000, rate=0.10),
            Tier(limit=float('inf'), rate=0.08),
        ),
    ),
    Institution(
        name="My Bank B",
        tiers=(
            Tier(limit=100_000, rate=0.11),
            Tier(limit=float('inf'), rate=0.09),
        ),
    ),
]

result = allocate(total=300_000, institutions=my_institutions)

print(f"\n=== Custom Institutions Test ===")
print(f"Total to allocate: $300,000")
print(f"Effective rate: {result.effective_rate:.2%}")
print(f"Expected annual return: ${result.expected_return:,.2f}")
print(f"\nAllocation:")

for name, amounts in result.allocations.items():
    total_alloc = sum(amounts)
    if total_alloc > 0:
        print(f"  {name}: ${total_alloc:,.0f}")
        for i, amt in enumerate(amounts):
            if amt > 0:
                inst = next(i for i in my_institutions if i.name == name)
                rate = inst.tiers[i].rate
                print(f"    Tier {i+1}: ${amt:,.0f} @ {rate:.1%}")

## Scenario 6: Edge Cases

Test boundary conditions.

In [ ]:
# Test 1: Zero allocation
result = allocate(total=0, institutions=institutions)
print(f"Zero allocation: ${result.expected_return:,.2f} return")

# Test 2: Very small amount
result = allocate(total=1_000, institutions=institutions)
print(f"\n1,000 MXN allocation:")
print(f"  Effective rate: {result.effective_rate:.2%}")
print(f"  Expected return: ${result.expected_return:,.2f}")

# Test 3: Exactly fills first tier
result = allocate(total=25_000, institutions=institutions)
print(f"\n25,000 MXN (fills Nu's first tier):")
print(f"  Effective rate: {result.effective_rate:.2%}")
print(f"  Expected return: ${result.expected_return:,.2f}")

# Test 4: Very large amount
result = allocate(total=10_000_000, institutions=institutions)
print(f"\n10,000,000 MXN allocation:")
print(f"  Effective rate: {result.effective_rate:.2%}")
print(f"  Expected return: ${result.expected_return:,.2f}")

## Try Your Own Tests

Modify the cell below to test any scenario you want!

In [ ]:
# Edit these values and run to test
test_amount = 150_000  # <-- Change this

result = allocate(total=test_amount, institutions=institutions)

print(f"\n=== Your Test: {test_amount:,} MXN ===")
print(f"Effective rate: {result.effective_rate:.2%}")
print(f"Expected annual return: ${result.expected_return:,.2f}")
print(f"\nAllocation by institution:")

for name, amounts in result.allocations.items():
    total_alloc = sum(amounts)
    if total_alloc > 0:
        pct = (total_alloc / test_amount) * 100
        print(f"  {name}: ${total_alloc:,.0f} ({pct:.1f}%)")
        for i, amt in enumerate(amounts):
            if amt > 0:
                inst = next(i for i in institutions if i.name == name)
                rate = inst.tiers[i].rate
                print(f"    Tier {i+1}: ${amt:,.0f} @ {rate:.1%}")